# Click-to-segment: training resultsInteractive object segmentation on ADE20K. The user clicks an object, the model returnsthat instance's mask. A UNet-style encoder-decoder trained from scratch, no pretrainedbackbone, taking 5 input channels: RGB plus a positive and a negative click map.This notebook reads the artefacts produced by `scripts/train_full.py` and reports whatthe trained model actually does.

In [ ]:
import json, sys, osfrom pathlib import PathREPO = Path.cwd()if REPO.name == "notebooks":    REPO = REPO.parentsys.path.insert(0, str(REPO))# $HOME differs between MetaCentrum nodes, so try the known locations rather# than trusting "~" to resolve to the storage the data actually lives on.CANDIDATE_ROOTS = [    Path("/storage/brno2/home") / os.environ.get("USER", "") / "projects/ade20k-reference/dataset/ADE20K_2021_17_01/images/ADE",    Path.home() / "projects/ade20k-reference/dataset/ADE20K_2021_17_01/images/ADE",]DATA_ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), None)OUTPUTS = REPO / "outputs"print("repo:     ", REPO)print("data root:", DATA_ROOT)print("outputs:  ", OUTPUTS)

## Training history

In [ ]:
with open(OUTPUTS / "history.json") as f:    results = json.load(f)# train_full.py writes a bare list while running and a summary dict at the end.history = results["history"] if isinstance(results, dict) else resultssummary = results if isinstance(results, dict) else {}epochs    = [h["epoch"]      for h in history]train_loss= [h["train_loss"] for h in history]val_loss  = [h["val_loss"]   for h in history]train_iou = [h["train_iou"]  for h in history]val_iou   = [h["val_iou"]    for h in history]best_epoch = summary.get("best_epoch") or epochs[val_iou.index(max(val_iou))]best_val   = summary.get("best_val_iou", max(val_iou))print(f"epochs trained : {len(history)}")print(f"best epoch     : {best_epoch}")print(f"best val IoU   : {best_val:.4f}")if "test_iou" in summary:    print(f"test IoU       : {summary['test_iou']:.4f}")print(f"mean epoch time: {sum(h['seconds'] for h in history)/len(history):.0f}s")

### Loss and IoU curvesThe dashed line marks the epoch with the best validation IoU. That epoch's weights arewhat `best.pt` holds and what the test set is scored against, so a late-training dipcannot contaminate the reported result.

In [ ]:
import matplotlib.pyplot as pltfig, (ax_loss, ax_iou) = plt.subplots(1, 2, figsize=(13, 4.5))ax_loss.plot(epochs, train_loss, label="train")ax_loss.plot(epochs, val_loss, label="validation")ax_loss.axvline(best_epoch, ls="--", c="gray", lw=1)ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("BCE + Dice loss")ax_loss.set_title("Loss"); ax_loss.legend(); ax_loss.grid(alpha=.3)ax_iou.plot(epochs, train_iou, label="train")ax_iou.plot(epochs, val_iou, label="validation")ax_iou.axvline(best_epoch, ls="--", c="gray", lw=1)ax_iou.scatter([best_epoch], [best_val], zorder=5, c="crimson",               label=f"best (epoch {best_epoch})")ax_iou.set_xlabel("epoch"); ax_iou.set_ylabel("IoU")ax_iou.set_title("Intersection over Union"); ax_iou.legend(); ax_iou.grid(alpha=.3)fig.tight_layout()plt.show()

### Reading the curvesTraining IoU keeps climbing while validation flattens, and the gap between them widensin the later epochs. That gap is the model beginning to memorise the training set ratherthan learning to generalise, which is why the best epoch is selected on validation andnot simply taken from the final epoch.The practical consequence: training for longer would not help. More data would.

## Results

In [ ]:
final = history[-1]best  = next(h for h in history if h["epoch"] == best_epoch)rows = [    ("train (final epoch)",      final["train_loss"], final["train_iou"]),    (f"validation (epoch {best_epoch})", best["val_loss"], best["val_iou"]),]if "test_iou" in summary:    rows.append(("test (held out)", summary["test_loss"], summary["test_iou"]))print(f"{'split':<28}{'loss':>10}{'IoU':>10}")print("-" * 48)for name, loss, iou in rows:    print(f"{name:<28}{loss:>10.4f}{iou:>10.4f}")if "splits" in summary:    print()    print("images per split:", summary["splits"])

The test score is the number that matters. It comes from the best-validation checkpointevaluated once on images the model never saw during training or epoch selection.Test IoU landing within a hundredth of validation IoU is the key sanity check: it saysthe 70/20/10 split is clean and that choosing the epoch on validation did not quietlyoverfit to it.

## Qualitative examplesNumbers hide the failure modes. Below are test-set predictions: the simulated click,the ground-truth mask, and what the model produced.

In [ ]:
import numpy as npimport torchimport yamlfrom src.data.ade20k import discover_samplesfrom src.data.dataset import ClickSegmentationDatasetfrom src.data.splits import split_image_pathsfrom src.model.unet import UNetfrom src.training.metrics import iou_scorewith open(REPO / "configs/train.yaml") as f:  train_cfg = yaml.safe_load(f)with open(REPO / "configs/clicks.yaml") as f: click_cfg = yaml.safe_load(f)["clicks"]device = torch.device("cuda" if torch.cuda.is_available() else "cpu")image_paths = discover_samples(DATA_ROOT)splits = split_image_paths(image_paths,                           ratios=tuple(train_cfg["training"]["splits"]),                           seed=train_cfg["training"]["split_seed"])test_ds = ClickSegmentationDataset(    splits["test"],    image_size=train_cfg["data"]["image_size"],    click_config=click_cfg,    deterministic=True,    lazy=True,    index_cache=OUTPUTS / "instance_index_test.json",)model = UNet(in_channels=5, out_channels=1,             base_channels=train_cfg["model"]["base_channels"]).to(device)ckpt = torch.load(OUTPUTS / "checkpoints/best.pt", map_location=device)model.load_state_dict(ckpt["model_state_dict"])model.eval()print(f"test instances: {len(test_ds)}   checkpoint from epoch {ckpt['epoch']}   device: {device}")

In [ ]:
N_SHOWN = 6rng = np.random.default_rng(0)picks = rng.choice(len(test_ds), size=N_SHOWN, replace=False)fig, axes = plt.subplots(N_SHOWN, 3, figsize=(10.5, 3.3 * N_SHOWN))for row, idx in enumerate(picks):    inputs, target = test_ds[int(idx)]    with torch.no_grad():        logits = model(inputs.unsqueeze(0).to(device))    pred = (torch.sigmoid(logits)[0, 0].cpu().numpy() > 0.5)    iou  = iou_score(logits.cpu(), target.unsqueeze(0))    image = inputs[:3].permute(1, 2, 0).numpy()    pos, neg = inputs[3].numpy(), inputs[4].numpy()    axes[row, 0].imshow(image)    for chan, colour, marker in ((pos, "lime", "+"), (neg, "red", "x")):        if chan.any():            ys, xs = np.nonzero(chan)            axes[row, 0].scatter(xs.mean(), ys.mean(), c=colour, marker=marker,                                 s=220, linewidths=3)    axes[row, 0].set_title("image + simulated click")    axes[row, 1].imshow(target[0].numpy(), cmap="gray")    axes[row, 1].set_title("ground truth")    axes[row, 2].imshow(pred, cmap="gray")    axes[row, 2].set_title(f"prediction (IoU {iou:.2f})")    for ax in axes[row]:        ax.axis("off")fig.tight_layout()plt.show()

Green `+` is the positive click identifying the target object; red `x`, where present, isa negative click just outside it.## Where this stands, and what would improve itThe model reliably finds the clicked object and gets its rough extent right. Errorsconcentrate in two places: boundaries are soft rather than crisp, and large objects withambiguous edges (road against sidewalk, building against sky) bleed into theirneighbours.Ranked by expected benefit:1. **More data.** Training used 3,000 of the 25,574 available images, about 12%. This is   the clearest limitation, and the curves support it: the model is data-starved rather   than capacity-starved or under-trained.2. **A pretrained encoder.** Training from scratch was a deliberate project requirement,   but an ImageNet-pretrained backbone is what most published interactive segmenters use   and would likely help most after data.3. **Higher resolution.** At 128x128 fine boundaries are barely representable. Only 0.24%   of instances vanish at this size, so this is about boundary precision, not coverage.4. **Iterative clicks.** Real users click repeatedly to correct a mask. Training with a   single click optimises for a task easier than the one the tool actually performs.Not yet measured: **NoC** (number of clicks to reach a target IoU), the standardinteractive-segmentation metric, and a comparison against a pretrained SAM baseline.